# Interview Question: Design and Implement a RAG System

Design and implement a Retrieval-Augmented Generation (RAG) system that meets the following requirements:

1. Document Ingestion:
    - The system should be able to load and process documents in PDF and text formats
    - Implement document chunking with appropriate overlap for context preservation

2. Vector Database:
    - Use FAISS as the vector store to save embeddings locally
    - Implement the all-MiniLM-L6-v2 model for generating embeddings
    - Design a solution to persist and reload the vector store efficiently

3. Retrieval and Generation:
    - Create a LangChain QA chain that retrieves relevant context and generates accurate answers
    - Implement similarity search with appropriate parameters
    - Ensure the system can handle follow-up questions maintaining context

4. Performance Considerations:
    - How would you optimize the system for large document collections?
    - What strategies would you use to handle documents with complex formatting?
    - How would you evaluate the quality of responses and improve retrieval precision?

Please implement a working prototype demonstrating these capabilities, explaining your design choices.

In [20]:
# Document Ingestion Class
# The class is used to laod teh documnets (.pdf, .text, .csv, .json)

import os
import datetime
from langchain.embeddings import OpenAIEmbeddings
from langchain_openai import AzureOpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.document_loaders import PyPDFLoader, TextLoader, CSVLoader, JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chat_models import ChatOpenAI, AzureChatOpenAI
from langchain.prompts import PromptTemplate
import faiss
import pandas as pd
import json
from typing import List, Dict, Any
from langchain.schema import Document
from dotenv import load_dotenv
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
load_dotenv()

True

In [27]:

class DocumentIngestion:
    def __init__(self, document_path, chunk_size=1000, chunk_overlap=200):
        self.document_path = document_path
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap,
            length_function=len
        )
        
    def load_document(self) -> List[Document]:
        """
        Load document based on file extension and return list of Document objects
        ready for embedding
        """
        _, file_extension = os.path.splitext(self.document_path)
        
        try:
            if file_extension.lower() == '.pdf':
                # Load PDF document using PyPDFLoader
                loader = PyPDFLoader(self.document_path)
                documents = loader.load()
                
            elif file_extension.lower() == '.txt':
                # Load text document using TextLoader
                loader = TextLoader(self.document_path)
                documents = loader.load()
                
            elif file_extension.lower() == '.csv':
                # Load CSV document - properly specify the CSV column names to use
                loader = CSVLoader(
                    file_path=self.document_path,
                    csv_args={
                        'delimiter': ',',
                        'quotechar': '"',
                    },
                    # By default, it creates a metadata column with 'source' file path
                )
                documents = loader.load()
                
            elif file_extension.lower() == '.json':
                # Load JSON document
                loader = JSONLoader(
                    file_path=self.document_path,
                    jq_schema=".",  # Extract data at root level
                    content_key=None,  # We'll use a custom extraction function
                    text_content=False,
                    json_lines=False
                )
                documents = loader.load()
            else:
                raise ValueError(f"Unsupported file format: {file_extension}")
            
            # Split documents into chunks for better context management
            split_documents = self.text_splitter.split_documents(documents)
            
            print(f"Loaded {len(documents)} document(s) and split into {len(split_documents)} chunks")
            return split_documents
            
        except Exception as e:
            print(f"Error loading document: {str(e)}")
            raise
    
    def process_documents_for_embedding(self) -> List[Document]:
        """
        Process documents to prepare them for embedding
        """
        # Load and chunk the document
        chunks = self.load_document()
        return chunks

class VectorStore:
    def __init__(self, embedding_model_name="text-embedding-ada-002", 
                 persist_directory="./vector_store"):
        
        self.embedding_model_name = embedding_model_name
        self.persist_directory = persist_directory
        
        # Initialize the embedding model with HuggingFaceEmbeddings
        self.embeddings = AzureOpenAIEmbeddings(azure_deployment=embedding_model_name)
        
        # Make sure the persist directory exists
        os.makedirs(self.persist_directory, exist_ok=True)
    
    def create_vector_store(self, documents):
        """
        Create a FAISS vector store from documents
        """
        vector_store = FAISS.from_documents(documents, self.embeddings)
        return vector_store
    
    def save_vector_store(self, vector_store, name=None):
        """
        Save the vector store to disk
        """
        if name is None:
            name = f"vectorstore"
            
        save_path = os.path.join(self.persist_directory, name)
        vector_store.save_local(save_path)
        print(f"Vector store saved to {save_path}")
        return save_path
    
    def load_vector_store(self, path):
        """
        Load a vector store from disk
        """
        if os.path.exists(path):
            vector_store = FAISS.load_local(path, self.embeddings)
            print(f"Vector store loaded from {path}")
            return vector_store
        else:
            raise FileNotFoundError(f"No vector store at {path}")
        

class RAGSystem:
    def __init__(self, model_name="gpt-35-turbo", temperature=0.0, embedding_model="text-embedding-ada-002"):
        self.model_name = model_name
        self.temperature = temperature
        self.embedding_model = embedding_model
        self.vector_store = None
        self.qa_chain = None
        self.conversation_history = []
        
    def ingest_document(self, document_path, chunk_size=1000, chunk_overlap=200):
        """
        Ingest a document, chunk it, and create vector store
        """
        # Create document ingestion instance with explicit chunk_size and chunk_overlap
        ingestion = DocumentIngestion(document_path, chunk_size=chunk_size, chunk_overlap=chunk_overlap)
        documents = ingestion.process_documents_for_embedding()
        
        # Create vector store
        vector_store_handler = VectorStore(embedding_model_name=self.embedding_model)
        self.vector_store = vector_store_handler.create_vector_store(documents)
        vector_store_path = vector_store_handler.save_vector_store(self.vector_store)
        
        return {
            "documents": len(documents),
            "vector_store_path": vector_store_path
        }
    
    def load_existing_vector_store(self, vector_store_path):
        """
        Load an existing vector store
        """
        vector_store_handler = VectorStore(embedding_model_name=self.embedding_model)
        self.vector_store = vector_store_handler.load_vector_store(vector_store_path)
    
    def setup_qa_chain(self):
        """
        Set up the QA chain with conversation history support for context-aware responses
        """
        if self.vector_store is None:
            raise ValueError("Vector store must be initialized before setting up QA chain")
            
        # Create a condense question prompt for follow-up questions
        condense_prompt = """Given the following conversation and a follow up question, 
        rephrase the follow up question to be a standalone question that captures all relevant context.

        Chat History:
        {chat_history}
        
        Follow Up Question: {question}
        
        Standalone Question:"""
        
        # Create the prompt for the final answer
        qa_prompt = """Use the following context to answer the question at the end.
        If you don't know the answer, just say you don't know. Don't try to make up an answer.
        
        Context: {context}
        
        Question: {question}
        
        Please provide a precise answer based only on the context above."""
            
        # Initialize the LLM
        model = AzureChatOpenAI(model_name=self.model_name, temperature=self.temperature)
        
        # Create memory object
        memory = ConversationBufferMemory(
            memory_key="chat_history",
            return_messages=True,
            output_key="answer"
        )
        
        # Set up conversational retrieval chain
        self.qa_chain = ConversationalRetrievalChain.from_llm(
            llm=model,
            retriever=self.vector_store.as_retriever(search_kwargs={"k": 4}),
            condense_question_llm=model,  # Use the same model for question condensing
            memory=memory,
            condense_question_prompt=PromptTemplate.from_template(condense_prompt),
            combine_docs_chain_kwargs={"prompt": PromptTemplate.from_template(qa_prompt)},
            return_source_documents=True,
            chain_type="stuff"
        )
    
    def clear_conversation_history(self):
        """
        Clear the conversation history and reset the memory in the QA chain
        """
        # Clear the internal conversation history list
        self.conversation_history = []
        
        # Reset the memory in the QA chain if it exists
        if self.qa_chain is not None and hasattr(self.qa_chain, 'memory'):
            self.qa_chain.memory.clear()
            print("Conversation history cleared successfully")
        else:
            print("No active QA chain with memory exists yet")
        
        return "Conversation history has been cleared."

    def ask(self, question):
        """
        Ask a question to the RAG system, maintaining conversation history
        """
        if self.qa_chain is None:
            self.setup_qa_chain()
        try:
            # Get answer with the condensed question based on history
            result = self.qa_chain({"question": question})
            
            # Update conversation history
            self.conversation_history.append({"role": "user", "content": question})
            self.conversation_history.append({"role": "assistant", "content": result["answer"]})
            
            return result["answer"]
        except Exception as e:
            error_message = f"Error in asking question: {str(e)}"
            print(error_message)
            return error_message


In [28]:
# Example usage with follow-up questions
if __name__ == "__main__":
    document_path = 'students.csv'
    rag_system = RAGSystem()

    rag_system.clear_conversation_history()
    
    # Ingest the document and create vector store
    ingest_result = rag_system.ingest_document(document_path, chunk_size=1000, chunk_overlap=200)
    print(f"Document ingestion result: {ingest_result}")
    
    # Set up QA chain
    rag_system.setup_qa_chain()
    
    # Ask initial question
    question1 = "What is the total and average marks of Student_12?"
    answer1 = rag_system.ask(question1)
    print(f"Q: {question1}")
    print(f"A: {answer1}\n")
    
    # Ask follow-up question
    question2 = "How does that compare to the class average and what was the class average?"
    answer2 = rag_system.ask(question2)
    print(f"Q: {question2}")
    print(f"A: {answer2}\n")
    
    # Another follow-up
    question3 = "Who has the highest marks in the class?"
    answer3 = rag_system.ask(question3)
    print(f"Q: {question3}")
    print(f"A: {answer3}")

No active QA chain with memory exists yet
Loaded 100 document(s) and split into 100 chunks
Vector store saved to ./vector_store\vectorstore
Document ingestion result: {'documents': 100, 'vector_store_path': './vector_store\\vectorstore'}
Q: What is the total and average marks of Student_12?
A: Total marks of Student_12: 460
Average marks of Student_12: 57.5

Q: How does that compare to the class average and what was the class average?
A: The total marks of Student_12 is 460 and the average marks is 64. The class average is 71.75. Therefore, Student_12 has a lower total and average marks compared to the class average.

Q: Who has the highest marks in the class?
A: Student_11 has the highest marks in the class. 

Student_12's total marks: 460
Student_12's average marks: 64.29

Class average total marks: 558
Class average average marks: 69.75

Student_12's total marks are lower than the class average, and Student_12's average marks are also lower than the class average.
